In [ ]:
# Load Data
import pandas as pd
quotation_data_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data.xlsx"
quotation_data_df = pd.read_excel(quotation_data_path)

In [ ]:
# Custom Date Feature Engineering
# Label missing Date, fill missing Date with median
# Extract Year, Quarter, Month, DayOfWeek and trasform them to cyclical sin and cos
# ================================================================
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class DateFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_column="Date"):
        self.date_column = date_column

    # fit method is only ever called on your training dataset, it learns the median.
    def fit(self, X, y=None):
        dates = pd.to_datetime(
            X[self.date_column],
            errors="coerce"
        )
        self.median_date_ = dates.dropna().median()
        return self
    
    # transform method is called on your training data and later on your hidden test data, it uses the learned median.
    def transform(self, X):
        X = X.copy()
        X[self.date_column] = pd.to_datetime(
            X[self.date_column],
            errors="coerce"
        )

        # Missing Indicator
        X["DateMissing"] = X[self.date_column].isna().astype(int)

        # Median Imputation
        X[self.date_column] = X[self.date_column].fillna(
            self.median_date_
        )

        # Date Features
        X["Date_Year"] = X[self.date_column].dt.year
        X["Date_Quarter"] = X[self.date_column].dt.quarter

        month = X[self.date_column].dt.month
        dow = X[self.date_column].dt.dayofweek

        X["month_sin"] = np.sin(2*np.pi*month/12)
        X["month_cos"] = np.cos(2*np.pi*month/12)

        X["dayofweek_sin"] = np.sin(2*np.pi*dow/7)
        X["dayofweek_cos"] = np.cos(2*np.pi*dow/7)

        X.drop(columns=self.date_column, inplace=True)
        return X

In [ ]:
# Numeric Feature Engineering
# Transform numeric feature using log to reduce skewness for Classification
# ================================================================
class NumericFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        # Reduce skewness
        X["Value"] = np.log1p(X["Value"])
        return X

In [ ]:
# Estimator Feature Engineering
# Split the values in the feature and create new columns based on that
# ================================================================
class EstimatorFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, column="Priced_By"):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Number of estimators
        X["EstimatorCount"] = (
            X[self.column]
            .str.split("/")
            .apply(len)
        )

        # If the value is "MISSING", treat it as 0 estimators
        X.loc[
            X[self.column].str.upper() == "MISSING",
            "EstimatorCount"
        ] = 0

        X["EstimatorCount"] = X["EstimatorCount"].astype(int)

        # Multiple estimator flag
        X["MultipleEstimators"] = (
            X["EstimatorCount"] > 1
        ).astype(int)

        # Missing Estimator flag
        X["EstimatorMissing"] = (
            X[self.column]
            .str.upper()
            .eq("MISSING")
        ).astype(int)

        return X

In [ ]:
# RareGrouper 
# ================================================================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, columns, min_frequency=10):
        self.columns = columns
        self.min_frequency = min_frequency
    
    def fit(self, X, y=None):
        X = X.copy()

        self.frequent_categories_ = {}

        for col in self.columns:

            counts = X[col].value_counts(dropna=False)

            self.frequent_categories_[col] = set(
                counts[counts >= self.min_frequency].index
            )

        return self
    
    def transform(self, X):
        X = X.copy()

        for col in self.columns:
            X[col] = np.where(
                X[col].isin(self.frequent_categories_[col]),
                X[col],
                "other"
            )

        return X
    
# Inspect the frequency distribution to determine the threshold for min frequency
client_counts = quotation_data_df["Client_Clean"].value_counts()

print(client_counts.describe())

print(client_counts.head(20))
print(client_counts.tail(20))

# Check how many grouper left with different min_frequency threshold
for threshold in [2, 5, 10, 20, 30, 50]:
    kept = (client_counts >= threshold).sum()
    print(f"{threshold:2d}: keep {kept:3d} clients")


In [ ]:
# Save dataframe after trasformation for checking
# ================================================================
from sklearn.pipeline import Pipeline

# 1. Build a preprocessing-only pipeline using your existing objects
debug_pipeline = Pipeline([
    ("date_features", DateFeatureTransformer()),
    ("rare_grouper", RareCategoryGrouper(
        columns=["Client_Clean", "Suburb"],
        min_frequency=20
    )),
    ("numeric_features", NumericFeatureTransformer()),
    ("preprocessor", preprocessor)
])

# 2. Fit and transform your training data to see the final numeric matrix
X_train_transformed = debug_pipeline.fit_transform(X_train)

# 3. Retrieve the post-one-hot-encoded column names from the preprocessor step
# Note: This requires scikit-learn 1.0 or newer
feature_names = debug_pipeline.named_steps["preprocessor"].get_feature_names_out()

# 4. Reconstruct into a beautiful, readable DataFrame
transformed_df = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)

# 5. Export locally to a CSV file
transformed_df.to_csv(r"C:\Users\Phong\Downloads\Quotation Data Test.csv", index=False)

print(f"✅ Success! Exported a matrix of {transformed_df.shape[0]} rows and {transformed_df.shape[1]} columns.")
print("Check your local directory for 'transformed_training_data.csv'.")

In [ ]:
# Save Trained Model and make prediction from local file - Classification
from joblib import dump

dump(lgr_best_model_v3, "LogisticRegression_V3.joblib")
print("Model saved.")

# Make Classification Prediction from data file
import pandas as pd
from joblib import load

# ==========================================
# Load trained model
# ==========================================
model = load("LogisticRegression_V3.joblib")

# ==========================================
# Load Excel file
# ==========================================
new_df = pd.read_excel(r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data Test.xlsx")

# ==========================================
# Keep only raw features
# ==========================================
raw_features = [
    "Date",
    "Value",
    "No_Wall_Types",
    "Priced_By",
    "Contact_Clean",
    "Suburb",
    "Client_Clean",
    "Timber_RW","RC_Pile","Steel_Beam","Sheetpile","Anchor","Block",
    "Shotcrete","Capping_Beam","Earthwork","Concrete_Slab","Precast",
    "Culvert","Slip_Repair","Soil_Nail","Rock_RW","Bridge","Concrete",
    "Design_and_Build","Budget","Drill_Only","Labour_Only",
    "Driven_Pile","Palisade","Boardwalk","Soldier","Insitu",
    "Barrier","Noise_RW","Base","Casing","Crib",
    "DayWork","Flood_Repair","Micro_Pile","Reno",
    "Temp_RW","Other"
]

X_new = new_df[raw_features]

# ==========================================
# Predict
# ==========================================
pred_class = model.predict(X_new)

pred_probability = model.predict_proba(X_new)

# Probability of Profit (class=1)
profit_probability = pred_probability[:,1]

loss_probability = pred_probability[:,0]

# ==========================================
# Save predictions
# ==========================================
new_df["Prediction"] = pred_class

new_df["Prediction_Label"] = new_df["Prediction"].map({
    0:"Lost",
    1:"Won"
})

new_df["Probability_Won"] = profit_probability

new_df["Probability_Lost"] = loss_probability

# ==========================================
# Save CSV
# ==========================================
output_file = r"C:\Users\Phong\Downloads\Quotation Prediction.csv"

new_df.to_csv(output_file, index=False)
print(f"Prediction saved to:\n{output_file}")

In [ ]:
# Analysis Data 01 - Grouping unique value of Feature 
# to calculate: "size" - total number for unique value of Feature in column "Success"
# to calculate: "mean" - average for unique value of Feature in column "Success" 
summary = (
    quotation_data_df
    .groupby("Priced_By")
    .agg(
        Quotes=("Success", "size"),
        WinRate=("Success", "mean")
    )
    .sort_values("Quotes", ascending=False)
)

print(summary)